# 00 · 环境准备与 GPU 检测

本 notebook 帮你把环境跑通,是后面所有实战的前提。目标:

1. 说明 **RTX 50 系 (Blackwell)** 的关键坑,以及正确的安装方式。
2. 检测 PyTorch / CUDA / 显卡是否可用。
3. 验证 **4-bit 量化 (bitsandbytes)** 是否可用(QLoRA 的前提)。

> 环境:Windows + RTX 5080 (16GB, Blackwell / sm_120) + Conda。

## 一、为什么 RTX 50 系要特别小心

RTX 50 系(5070/5080/5090)是 **Blackwell 架构**,计算能力 **sm_120**。
很多稳定版 PyTorch 的预编译包里**没有**针对 sm_120 编译的 CUDA kernel,于是一运行就报:

```
CUDA error: no kernel image is available for execution on the device
```

**正确做法:安装 CUDA 12.8 (cu128) 的 PyTorch(≥ 2.7)。**

在已激活的 `test-lora` conda 环境里执行(只需一次):

```bash
# 1) Blackwell 专用 PyTorch(务必用 cu128 源,不要用普通 pip install torch)
pip install --index-url https://download.pytorch.org/whl/cu128 torch torchvision

# 2) 其余依赖
pip install -r ../requirements.txt
```

`bitsandbytes` 也要用较新版本(≥ 0.45),它的 4-bit kernel 才支持 Blackwell,这是 QLoRA 能跑的前提。

## 二、检测 PyTorch 与显卡

运行下面这格,确认 CUDA 可用、能识别到你的 5080,并且计算能力是 (12, 0)。

In [ ]:
import torch

print(f"PyTorch 版本      : {torch.__version__}")
print(f"编译时 CUDA 版本  : {torch.version.cuda}")
print(f"CUDA 是否可用     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    idx = torch.cuda.current_device()
    cap = torch.cuda.get_device_capability(idx)
    total_gb = torch.cuda.get_device_properties(idx).total_memory / 1024**3
    print(f"显卡名称          : {torch.cuda.get_device_name(idx)}")
    print(f"计算能力 (sm)     : {cap[0]}.{cap[1]}  (Blackwell/50系应为 12.0)")
    print(f"总显存            : {total_gb:.1f} GB")

    # Blackwell 用户的自检:确认真的能在 GPU 上做运算,而不是加载时不报错、运算时才崩
    try:
        x = torch.randn(1024, 1024, device="cuda")
        y = (x @ x).sum().item()
        print("GPU 实际运算测试  : 通过 ✅")
    except Exception as e:
        print("GPU 实际运算测试  : 失败 ❌ —— 极可能是 PyTorch 没装 cu128 版本")
        print(f"  错误信息: {e}")
else:
    print("未检测到可用 CUDA,请检查驱动与 cu128 版 PyTorch 是否装好。")

## 三、验证 4-bit 量化 (bitsandbytes) 可用

QLoRA 依赖 `bitsandbytes` 的 4-bit kernel。下面这格做一个最小的 4-bit 量化线性层前向,能跑通说明 QLoRA 环境就绪。

In [ ]:
import importlib.metadata as md

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes 版本 : {md.version('bitsandbytes')}")

    # 用一个 4-bit 线性层做一次前向,验证 Blackwell 上 4-bit kernel 真能用
    import torch
    layer = bnb.nn.Linear4bit(64, 32, bias=False, compute_dtype=torch.float16).cuda()
    out = layer(torch.randn(4, 64, device="cuda", dtype=torch.float16))
    print(f"4-bit 前向输出形状: {tuple(out.shape)}")
    print("4-bit 量化测试    : 通过 ✅  (QLoRA 环境就绪)")
except Exception as e:
    print("4-bit 量化测试    : 失败 ❌")
    print(f"  错误信息: {e}")
    print("  提示: 请确认 bitsandbytes>=0.45,且 PyTorch 为 cu128 版本。")

## 四、检查其余核心依赖版本

确认 transformers / peft / trl / datasets / accelerate 都装好了。

In [ ]:
import importlib.metadata as md

for pkg in ["transformers", "peft", "trl", "datasets", "accelerate"]:
    try:
        print(f"{pkg:<14}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<14}: 未安装 ❌ —— 请执行 pip install -r ../requirements.txt")

## 小结

- 上面三个测试(GPU 运算、4-bit 量化、依赖版本)都通过后,环境就准备好了。
- 如果 GPU 运算测试失败,几乎都是 **PyTorch 没装 cu128 版本**,重装即可:

```bash
pip uninstall -y torch torchvision
pip install --index-url https://download.pytorch.org/whl/cu128 torch torchvision
```

接下来进入 [`01_PEFT与LoRA原理与实战`](01_PEFT与LoRA原理与实战.ipynb),开始正式学习 LoRA。